# 실습 3주차: 학습 루프를 직접 만든다

> **시나리오 — 오늘 만들 것**
>
>
> 지난주에 **뜻도 모르고 그대로 옮겨 적은 네 줄**이 있다.
>
> ```python
> optimizer.zero_grad()
> loss = criterion(model(xb), yb)
> loss.backward()
> optimizer.step()
> ```
>
> 오늘 이 네 줄을 **한 줄씩 연다.** 그리고 그 루프로
> **펭귄 4개 치수 → 3종(Adelie / Chinstrap / Gentoo) 분류**를 학습시킨다.
> 오늘부터는 회귀가 아니라 **분류**다.
>
> - **대응 이론**: [Ch03 딥러닝의 학습: 손실함수와 경사하강법](ch03.qmd)
> - **계산 약속**: 이 수업에서 $\log$ 는 항상 **자연로그** $\ln$ 이다.


> **오늘 배우는 PyTorch 부품**
>
>
> | 부품 | 네 줄 중 어디 |
> |------|------|
> | `nn.MSELoss` · `nn.CrossEntropyLoss` | ② `criterion(...)` |
> | `requires_grad` · `.backward()` · `.grad` | ③ `loss.backward()` |
> | `torch.optim.SGD` · `.step()` | ④ `optimizer.step()` |
> | `.zero_grad()` | ① 왜 지워야 하는가 |
> | `softmax` | 분류에서 점수를 확률로 |

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

---

# 1. 손실 — 얼마나 틀렸는지 숫자 하나로

## 1-1. 회귀의 손실: MSE

In [ ]:
y_true = torch.tensor([3.0, -0.5, 2.0, 7.0])
y_pred = torch.tensor([2.5,  0.0, 2.0, 8.0])

mse_manual = ((y_true - y_pred) ** 2).mean()
mse_torch = nn.MSELoss()(y_pred, y_true)

print('오차     :', (y_true - y_pred).numpy())
print('직접 계산:', float(mse_manual))
print('nn.MSELoss:', float(mse_torch))

## 1-2. 분류에서는 MSE를 쓰지 않는다

분류의 정답은 "3번 클래스"처럼 **번호**다. 번호끼리 빼는 것은 의미가 없다
(1번과 3번이 1번과 2번보다 "두 배 틀린" 것이 아니다).

그래서 분류에서는 모형이 **확률**을 내게 하고, 그 확률로 손실을 만든다.

## 1-3. Softmax — 점수를 확률로

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

In [ ]:
z = torch.tensor([2.0, 1.0, 0.1])          # 모형이 낸 점수(로짓) 3개

e = torch.exp(z)
p_manual = e / e.sum()

print('로짓  :', z.numpy())
print('e^z   :', e.numpy().round(4))
print('합    :', round(float(e.sum()), 4))
print('확률  :', p_manual.numpy().round(4))
print('확률 합:', round(float(p_manual.sum()), 6))
print('\ntorch :', z.softmax(dim=0).numpy().round(4))

> **직접 해보기 ① — Softmax를 직접 만들기**
>
>
> `softmax(z)` 함수를 작성하시오. (힌트: `torch.exp`, `.sum()`)

In [ ]:
# ✏️ 직접 채워 보세요
def softmax(z):
    return None            # ← 여기를 채우세요

got = softmax(torch.tensor([1.0, 2.0, 3.0]))
assert got is not None, '아직 채우지 않았습니다'
assert abs(float(got.sum()) - 1.0) < 1e-5, f'합이 1이 아닙니다: {float(got.sum())}'
print('통과', got.numpy().round(4))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
def softmax(z):
    e = torch.exp(z)
    return e / e.sum()

print(softmax(torch.tensor([1.0, 2.0, 3.0])).numpy().round(4))

## 1-4. 교차 엔트로피 — 정답 확률만 본다

$$\mathcal{L} = -\log p_{\text{정답}}$$

In [ ]:
p = z.softmax(dim=0)
label = 0                                   # 정답이 0번 클래스라면

ce_manual = -torch.log(p[label])
ce_torch = nn.CrossEntropyLoss()(z.unsqueeze(0), torch.tensor([label]))

print('정답 클래스의 확률:', round(float(p[label]), 4))
print('직접 계산 -log(p) :', round(float(ce_manual), 4))
print('nn.CrossEntropyLoss:', round(float(ce_torch), 4))

In [ ]:
# 정답 확률이 높을수록 손실이 작다
for q in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f'정답 확률 {q:5.2f}  →  손실 {-np.log(q):6.3f}')

> **`nn.CrossEntropyLoss` 의 두 가지 함정**
>
>
> **① Softmax를 미리 적용하면 안 된다.** 이 함수가 **내부에서 Softmax를 한다.**
> 확률을 넣으면 Softmax가 두 번 걸린다.
>
> **② 정답은 원-핫이 아니라 클래스 번호(정수)** 다. `dtype` 은 `long` 이어야 한다.

In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])
target = torch.tensor([0, 1])                 # 클래스 번호 (원-핫이 아니다)

print('입력 shape :', tuple(logits.shape), '  정답 shape :', tuple(target.shape), target.dtype)
print('손실       :', round(float(nn.CrossEntropyLoss()(logits, target)), 4))
print('\n두 건을 따로 계산해 평균:',
      round(float((-torch.log(logits[0].softmax(0)[0]) - torch.log(logits[1].softmax(0)[1])) / 2), 4))

> **직접 해보기 ② — 교차 엔트로피를 직접 만들기**
>
>
> 로짓 `z`(1차원)와 정답 번호 `label` 을 받아 손실을 돌려주는 `cross_entropy(z, label)` 을 작성하시오.

In [ ]:
# ✏️ 직접 채워 보세요
def cross_entropy(z, label):
    return None            # ← 여기를 채우세요

got = cross_entropy(torch.tensor([2.0, 1.0, 0.1]), 0)
assert got is not None, '아직 채우지 않았습니다'
assert abs(float(got) - 0.4170) < 1e-3, f'값이 다릅니다: {float(got)}'
print('통과', round(float(got), 4))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
def cross_entropy(z, label):
    return -torch.log(softmax(z)[label])

print(round(float(cross_entropy(torch.tensor([2.0, 1.0, 0.1]), 0)), 4))

---

# 2. 기울기 — 어느 쪽으로 가야 하나

## 2-1. `requires_grad` 와 `backward()`

In [ ]:
w = torch.tensor(3.0, requires_grad=True)     # 이 값에 대한 기울기를 추적한다

L = (w - 1.0) ** 2                            # 손실 = (w-1)^2
L.backward()                                  # 기울기 계산

print('손실 L      :', float(L))
print('기울기 dL/dw:', float(w.grad))
print('손으로      : 2(w-1) = 2(3-1) =', 2 * (3.0 - 1.0))

## 2-2. 수치미분과 대조

미분의 정의로도 확인해 본다.

$$\frac{dL}{dw} \approx \frac{L(w+h) - L(w-h)}{2h}$$

In [ ]:
f = lambda v: (v - 1.0) ** 2
h = 1e-4
numeric = (f(3.0 + h) - f(3.0 - h)) / (2 * h)

print('자동미분:', float(w.grad))
print('수치미분:', round(numeric, 6))

## 2-3. 기울기는 **쌓인다**

In [ ]:
w = torch.tensor(3.0, requires_grad=True)

for step in range(3):
    L = (w - 1.0) ** 2
    L.backward()
    print(f'{step+1}번째 backward 후 w.grad = {float(w.grad)}')

In [ ]:
w = torch.tensor(3.0, requires_grad=True)

for step in range(3):
    if w.grad is not None:
        w.grad.zero_()                  # 매번 지운다
    L = (w - 1.0) ** 2
    L.backward()
    print(f'{step+1}번째 (지우고) w.grad = {float(w.grad)}')

> **`zero_grad()` 를 빼먹으면 조용히 틀린다**
>
>
> 에러는 안 나고 기울기만 계속 커진다. 학습이 이상하면 여기부터 확인한다.


> **직접 해보기 ③ — 기울기 확인하기**
>
>
> $L = w^3 + 2w$ 일 때 $w=2$ 에서의 기울기를 자동미분으로 구하고,
> 손으로 구한 $3w^2+2$ 와 맞는지 확인하시오.

In [ ]:
# ✏️ 직접 채워 보세요
w = torch.tensor(2.0, requires_grad=True)
L = None                       # ← 손실을 정의하세요
L.backward()

assert abs(float(w.grad) - 14.0) < 1e-4, f'값이 다릅니다: {float(w.grad)}'
print('통과  자동미분', float(w.grad), ' 손계산', 3*2**2 + 2)

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
w = torch.tensor(2.0, requires_grad=True)
L = w ** 3 + 2 * w
L.backward()
print('자동미분', float(w.grad), ' 손계산 3w²+2 =', 3 * 2 ** 2 + 2)

---

# 3. 경사하강 — 기울기의 반대로 한 걸음

$$w \leftarrow w - \eta \frac{\partial L}{\partial w}$$

## 3-1. 한 스텝을 손으로

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
lr = 0.1

L = (w - 1.0) ** 2
L.backward()
g = float(w.grad)

print('현재 w   :', 3.0)
print('손실     :', float(L))
print('기울기   :', g)
print('갱신 후 w:', 3.0 - lr * g, '  = 3 - 0.1 x', g)

## 3-2. 반복하면 학습이다

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
path = [float(w)]

for step in range(30):
    if w.grad is not None:
        w.grad.zero_()
    L = (w - 1.0) ** 2
    L.backward()
    with torch.no_grad():
        w -= lr * w.grad            # 갱신
    path.append(float(w))

print('시작 w :', path[0])
print('끝 w   :', round(path[-1], 6), '  (정답은 1)')

plt.figure(figsize=(5.6, 3.4))
plt.plot(path, 'o-', ms=3)
plt.axhline(1.0, color='red', ls='--', label='정답 w = 1')
plt.xlabel('step'); plt.ylabel('w'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **직접 해보기 ④ — 학습률을 바꿔 보기**
>
>
> `lr` 을 `0.01`, `0.5`, `1.1` 로 바꿔 30스텝씩 돌리고, `w` 가 어떻게 움직이는지 겹쳐 그리시오.
> `1.1` 에서는 무슨 일이 일어나는가?

In [ ]:
# ✏️ 직접 채워 보세요
plt.figure(figsize=(6, 3.6))
for lr_try in [0.01, 0.5, 1.1]:
    w = torch.tensor(3.0, requires_grad=True)
    path = [float(w)]
    for step in range(30):
        ...                          # ← 위 3-2의 루프를 옮겨 오세요
    plt.plot(path, label=f'lr = {lr_try}')
plt.axhline(1.0, color='red', ls='--')
plt.xlabel('step'); plt.ylabel('w'); plt.legend(); plt.show()

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
plt.figure(figsize=(6, 3.6))
for lr_try in [0.01, 0.5, 1.1]:
    w = torch.tensor(3.0, requires_grad=True)
    path = [float(w)]
    for step in range(30):
        if w.grad is not None:
            w.grad.zero_()
        L = (w - 1.0) ** 2
        L.backward()
        with torch.no_grad():
            w -= lr_try * w.grad
        path.append(float(w))
    plt.plot(path, label=f'lr = {lr_try}  (끝: {path[-1]:.2f})')
plt.axhline(1.0, color='red', ls='--')
plt.xlabel('step'); plt.ylabel('w')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

학습률이 작으면 **너무 느리고**, 크면 **최솟값을 뛰어넘어 발산**한다.

---

# 4. 학습 루프의 뼈대

## 4-1. 네 줄

```python
optimizer.zero_grad()          # ① 지난 기울기를 지운다
loss = criterion(model(X), y)  # ② 예측하고 손실을 잰다
loss.backward()                # ③ 기울기를 구한다
optimizer.step()               # ④ 파라미터를 갱신한다
```

## 4-2. optimizer 없이 직접 갱신해 보기

In [ ]:
torch.manual_seed(0)
toy = nn.Linear(2, 1)
Xd = torch.randn(20, 2)
yd = (Xd @ torch.tensor([2.0, -1.0]) + 0.5).unsqueeze(1)

crit = nn.MSELoss()
for ep in range(200):
    loss = crit(toy(Xd), yd)
    toy.zero_grad()
    loss.backward()
    with torch.no_grad():
        for p in toy.parameters():
            p -= 0.1 * p.grad              # ← optimizer.step() 이 하는 일
print('직접 갱신 : w =', toy.weight.detach().numpy().round(3),
      ' b =', round(float(toy.bias), 3), ' 손실', round(float(loss), 6))

## 4-3. optimizer로 같은 일

In [ ]:
torch.manual_seed(0)
toy2 = nn.Linear(2, 1)
opt = torch.optim.SGD(toy2.parameters(), lr=0.1)

for ep in range(200):
    opt.zero_grad()
    loss = crit(toy2(Xd), yd)
    loss.backward()
    opt.step()
print('optimizer : w =', toy2.weight.detach().numpy().round(3),
      ' b =', round(float(toy2.bias), 3), ' 손실', round(float(loss), 6))

**같은 결과다.** `optimizer` 는 갱신 규칙을 대신 적어 주는 도구일 뿐이다.

---

# 5. 완성 — 펭귄 3종 분류

## 5-1. 데이터 준비

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
d = pd.read_csv(URL).dropna().reset_index(drop=True)

cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
classes = sorted(d['species'].unique())

X = torch.tensor(d[cols].to_numpy(dtype='float32'))
y = torch.tensor(d['species'].map({c: i for i, c in enumerate(classes)}).to_numpy())

print('클래스 :', classes)
print('X', tuple(X.shape), '  y', tuple(y.shape), y.dtype)
print('클래스별 수:', torch.bincount(y).tolist())

> **분류의 정답은 **정수 클래스 번호**다**
>
>
> 회귀에서는 `y` 를 `(n, 1)` 로 세웠지만, 분류에서는 **`(n,)` 짜리 `long` 텐서**다.
> `nn.CrossEntropyLoss` 가 그 형태를 요구한다.

In [ ]:
# 숫자 크기를 맞춰 준다 (4주차에 제대로 배운다)
X = (X - X.mean(0)) / X.std(0)

ds = TensorDataset(X, y)
loader = DataLoader(ds, batch_size=len(ds))     # 오늘도 전부 한 번에

print('데이터 수:', len(ds), '  에폭당 배치 수:', len(loader))
xb, yb = next(iter(loader))
print('한 배치 :', tuple(xb.shape), tuple(yb.shape))

## 5-2. 모형

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(4, 16), nn.ReLU(),
    nn.Linear(16, 3),               # 출력 3개 = 클래스 3개 (Softmax는 손실 안에)
)
print(model)
print('파라미터:', sum(p.numel() for p in model.parameters()))

> 출력층에 **Softmax를 붙이지 않는다.** `nn.CrossEntropyLoss` 가 내부에서 하기 때문이다.
> 모형은 **로짓**을 내놓는다.


## 5-3. 표준 학습 루프

이제 네 줄의 뜻을 전부 안다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

hist = {'loss': [], 'acc': []}
for epoch in range(300):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()                   # ① 지난 기울기를 지운다
        loss = criterion(model(xb), yb)         # ② 예측하고 손실을 잰다
        loss.backward()                         # ③ 기울기를 구한다
        optimizer.step()                        # ④ 파라미터를 갱신한다

    model.eval()
    with torch.no_grad():
        out = model(X)
        hist['loss'].append(float(criterion(out, y)))
        hist['acc'].append(float((out.argmax(1) == y).float().mean()))

print('첫 손실  :', round(hist['loss'][0], 4))
print('마지막   :', round(hist['loss'][-1], 4))
print('정확도   :', round(hist['acc'][-1], 4))

## 5-4. 러닝커브 — 손실과 지표를 **둘 다**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(hist['loss']); axes[0].set_ylabel('cross entropy'); axes[0].set_yscale('log')
axes[1].plot(hist['acc'], color='C2'); axes[1].set_ylabel('accuracy')
for ax in axes:
    ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **손실과 지표는 다르다**
>
>
> 학습은 **손실**(교차 엔트로피)로 하고, 보고는 **정확도**로 한다.
> 손실은 미분 가능해야 하지만 정확도는 미분할 수 없기 때문이다.


## 5-5. 클래스별로 나누어 본다

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(X).argmax(1)

cm = pd.crosstab(pd.Series(y.numpy(), name='실제'), pd.Series(pred.numpy(), name='예측'))
cm.index = [classes[i] for i in cm.index]
cm.columns = [classes[i] for i in cm.columns]
print(cm, '\n')

for i, c in enumerate(classes):
    m = y == i
    print(f'{c:10s} {int(m.sum()):3d}마리 중 {int((pred[m]==i).sum()):3d}마리 정답  '
          f'({float((pred[m]==i).float().mean()):.3f})')

> **이 정확도를 믿어도 되는가**
>
>
> 우리는 **333마리로 학습하고, 같은 333마리로 정확도를 쟀다.**
> 시험 범위를 미리 보고 시험을 친 셈이다.
>
> 이 숫자가 왜 위험한지, 어떻게 재야 하는지는 **4주차**에서 정면으로 다룬다.


> **직접 해보기 ⑤ — 은닉층을 없애면**
>
>
> `nn.Linear(4, 3)` 한 층만으로 같은 학습을 돌려 정확도를 비교하시오.
> 이 문제에도 은닉층이 필요한가?

In [ ]:
# ✏️ 직접 채워 보세요
torch.manual_seed(42)
simple = None                  # ← nn.Linear(...)
# 위 5-3의 루프를 그대로 옮겨 300에폭 학습시키고 정확도를 출력하세요

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
torch.manual_seed(42)
simple = nn.Linear(4, 3)
opt2 = torch.optim.SGD(simple.parameters(), lr=0.1)
for epoch in range(300):
    for xb, yb in loader:
        opt2.zero_grad()
        criterion(simple(xb), yb).backward()
        opt2.step()
with torch.no_grad():
    acc = float((simple(X).argmax(1) == y).float().mean())
print('1층(선형) 파라미터', sum(p.numel() for p in simple.parameters()), ' 정확도', round(acc, 4))
print('2층 MLP   파라미터', sum(p.numel() for p in model.parameters()), ' 정확도', round(hist['acc'][-1], 4))

이 문제는 **직선으로도 거의 갈라진다.** 은닉층이 항상 이득은 아니다 —
2주차 자동차 연비처럼 **휘어 있는 문제**에서 비로소 값을 한다.

---

# 6. 정리

> **이번 주 체크포인트**
>
>
> | 하고 싶은 일 | 코드 |
> |------|------|
> | 회귀 손실 | `nn.MSELoss()(pred, target)` |
> | 점수 → 확률 | `z.softmax(dim=-1)` |
> | 분류 손실 | `nn.CrossEntropyLoss()(logits, target)` — **로짓** + **정수 라벨** |
> | 기울기 추적 | `torch.tensor(..., requires_grad=True)` |
> | 기울기 계산 | `loss.backward()` → `p.grad` |
> | 기울기 지우기 | `model.zero_grad()` / `optimizer.zero_grad()` |
> | 갱신 | `p -= lr * p.grad` 또는 `optimizer.step()` |
> | 정확도 | `(logits.argmax(1) == y).float().mean()` |
> | 혼동행렬 | `pd.crosstab(실제, 예측)` |


**학습 루프 네 줄**

```python
optimizer.zero_grad()
loss = criterion(model(X), y)
loss.backward()
optimizer.step()
```

CNN도, Transformer도 이 네 줄을 반복한다. 바뀌는 것은 `model` 뿐이다.

## 스스로 확인해 보기

아래 결과를 먼저 예상한 뒤 실행해서 대조한다.

In [ ]:
z = torch.tensor([[1.0, 2.0, 3.0]])
t = torch.tensor([2])

print('softmax        :', z.softmax(1).numpy().round(4))
print('정답 확률      :', round(float(z.softmax(1)[0, 2]), 4))
print('-log(정답확률) :', round(float(-torch.log(z.softmax(1)[0, 2])), 4))
print('CrossEntropy   :', round(float(nn.CrossEntropyLoss()(z, t)), 4))

a = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(2.0, requires_grad=True)
L = (a * b + a) ** 2
L.backward()
print('\ndL/da :', float(a.grad), '  손계산 2(ab+a)(b+1) =', 2*(1*2+1)*(2+1))
print('dL/db :', float(b.grad), '  손계산 2(ab+a)(a)    =', 2*(1*2+1)*1)

---

## 다음 실습

[실습 4주차: 진짜 표 하나를 끝까지](lab04.qmd) —
결측치와 범주형이 섞인 표를 **순서대로** 처리하고, 미니배치로 학습하고,
러닝커브로 진단해 조기 종료까지 한다.